<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_CRCV_Securitization_Non_ACTP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# --- Helper function for display ---
def display_df(df, title=""):
    """Prints a DataFrame with a title for clear output."""
    print(f"--- {title} ---")
    # Use to_string() to ensure the full DataFrame is printed
    print(df.to_string())
    print("\n" + "="*70 + "\n")

# ==============================================================================
# Cell 1: Steps 3 & 4 - Establish Gross and Net Positions
# ==============================================================================
print("### Steps 3 & 4: Establish Gross and Net Positions ###\n")
print("Each tranche is a unique risk factor, so no netting is required.\n")

# Define the portfolio data
data = {
    'Bucket': [3, 3, 3, 14, 14, 14],
    'Risk Factor': ['RMBS Tranche A', 'RMBS Tranche B', 'RMBS Tranche C', 'ABS Tranche 1', 'ABS Tranche 2', 'ABS Tranche 3'],
    'CVR+': [750, -400, 250, -800, 300, -100],
    'CVR-': [-600, 550, -150, 950, -200, 450]
}

net_positions_df = pd.DataFrame(data).set_index(['Bucket', 'Risk Factor'])

display_df(net_positions_df, "Final Net Positions")


# ==============================================================================
# Cell 2: Steps 5 & 6 - Determine Base Curvature Correlations (Medium Scenario)
# ==============================================================================
print("### Steps 5 & 6: Determine Base Curvature Correlations (Medium Scenario) ###\n")
print("Curvature correlations are the square of the corresponding delta correlations (Article 325ay(5)).\n")

# --- Intra-Bucket Correlation (rho) ---
# Delta correlation between different tranches within the same Non-ACTP bucket is 40% (Article 325an(1))
delta_rho_kl = 0.40
curvature_rho_medium = delta_rho_kl**2
print(f"Base Intra-Bucket Delta Correlation (rho_delta): {delta_rho_kl:.2%}")
print(f"Base Intra-Bucket Curvature Correlation (rho_curvature): {curvature_rho_medium:.4f} or {curvature_rho_medium:.2%}\n")

# --- Cross-Bucket Correlation (gamma) ---
# Delta correlation between different Non-ACTP buckets is 0% (Article 325ao)
delta_gamma_bc = 0.0
curvature_gamma_medium = delta_gamma_bc**2
print(f"Base Cross-Bucket Delta Correlation (gamma_delta): {delta_gamma_bc:.2%}")
print(f"Base Cross-Bucket Curvature Correlation (gamma_curvature): {curvature_gamma_medium:.4f} or {curvature_gamma_medium:.2%}")
print("\n" + "="*70 + "\n")

# ==============================================================================
# Cell 3: Step 7 & 8 - Bucket Capital Calculation Logic
# ==============================================================================
print("### Steps 7 & 8: Bucket-Level Capital Calculation (Medium Scenario) ###\n")

def psi(val1, val2):
    """Safeguard function from Article 325g(4)."""
    return 0 if val1 < 0 and val2 < 0 else 1

def calculate_bucket_capital(bucket_df, correlation):
    """
    Calculates K_b+, K_b-, the final K_b, and the selected scenario for a single bucket.
    Implements the logic from Article 325g(4).
    """
    # --- Upward Scenario (K_b+) ---
    cvr_plus = bucket_df['CVR+']
    sum_sq_cvr_plus = np.sum(np.maximum(cvr_plus, 0)**2)

    correlation_term_plus = 0
    # Iterate through all unique pairs of risk factors
    for i in range(len(cvr_plus)):
        for j in range(i + 1, len(cvr_plus)):
            cvr_k_plus = cvr_plus.iloc[i]
            cvr_l_plus = cvr_plus.iloc[j]
            correlation_term_plus += 2 * correlation * cvr_k_plus * cvr_l_plus * psi(cvr_k_plus, cvr_l_plus)

    kb_plus = np.sqrt(max(0, sum_sq_cvr_plus + correlation_term_plus))

    # --- Downward Scenario (K_b-) ---
    cvr_minus = bucket_df['CVR-']
    sum_sq_cvr_minus = np.sum(np.maximum(cvr_minus, 0)**2)

    correlation_term_minus = 0
    # Iterate through all unique pairs of risk factors
    for i in range(len(cvr_minus)):
        for j in range(i + 1, len(cvr_minus)):
            cvr_k_minus = cvr_minus.iloc[i]
            cvr_l_minus = cvr_minus.iloc[j]
            correlation_term_minus += 2 * correlation * cvr_k_minus * cvr_l_minus * psi(cvr_k_minus, cvr_l_minus)

    kb_minus = np.sqrt(max(0, sum_sq_cvr_minus + correlation_term_minus))

    # --- Final K_b and Scenario Selection ---
    kb_final = max(kb_plus, kb_minus)
    selected_scenario = 'Upward' if kb_plus > kb_minus else 'Downward'

    # Per Article 325g(4), handle the tie-breaking case
    if kb_plus == kb_minus:
        selected_scenario = 'Upward' if cvr_plus.sum() > cvr_minus.sum() else 'Downward'

    return kb_plus, kb_minus, kb_final, selected_scenario

# Calculate for each bucket using medium correlation
bucket_results_medium = {}
for bucket_id, group in net_positions_df.groupby(level='Bucket'):
    print(f"Calculating for Bucket {bucket_id}...")
    k_plus, k_minus, k_final, scenario = calculate_bucket_capital(group, curvature_rho_medium)
    bucket_results_medium[bucket_id] = {
        'K_b+': k_plus,
        'K_b-': k_minus,
        'K_b': k_final,
        'Selected Scenario': scenario
    }
    print(f"  K_b+ = {k_plus:,.2f}")
    print(f"  K_b- = {k_minus:,.2f}")
    print(f"  Final K_b = {k_final:,.2f}")
    print(f"  Selected Scenario: {scenario}\n")

display_df(pd.DataFrame(bucket_results_medium).T, "Summary of Bucket Capital (Medium Scenario)")

# ==============================================================================
# Cell 4: Step 9 - Determine Bucket Sums (S_b)
# ==============================================================================
print("### Step 9: Determine Bucket Sums (S_b) ###\n")
print("Calculating S_b based on the selected scenario for each bucket.\n")

bucket_sums_medium = {}
for bucket_id, result in bucket_results_medium.items():
    bucket_df = net_positions_df.loc[bucket_id]
    if result['Selected Scenario'] == 'Upward':
        s_b = bucket_df['CVR+'].sum()
    else:
        s_b = bucket_df['CVR-'].sum()
    bucket_sums_medium[bucket_id] = s_b

s_b_df = pd.DataFrame.from_dict(bucket_sums_medium, orient='index', columns=['S_b'])
display_df(s_b_df, "Bucket Sums (S_b) for Medium Scenario")

# ==============================================================================
# Cell 5: Step 10 - Calculate Cross-Bucket Capital (Medium Scenario)
# ==============================================================================
print("### Step 10: Calculate Cross-Bucket Capital (Medium Scenario) ###\n")
print("Since cross-bucket correlation is 0, this simplifies to the root of the sum of squares.\n")

# Sum of squared K_b values
sum_sq_kb = sum(res['K_b']**2 for res in bucket_results_medium.values())

# For this risk class, the cross-bucket correlation term is zero
cross_bucket_corr_term = 0

# Final RCCR for medium scenario
rccr_medium = np.sqrt(max(0, sum_sq_kb + cross_bucket_corr_term))

print(f"Sum of Squared K_b's = {sum_sq_kb:,.2f}")
print(f"Cross-Bucket Correlation Term = {cross_bucket_corr_term:,.2f}")
print(f"Final Capital (Medium Scenario) = sqrt({sum_sq_kb:,.0f}) = {rccr_medium:,.2f}")
print("\n" + "="*70 + "\n")


# ==============================================================================
# Cell 6: Step 11 - Correlation Scenarios & Final Capital
# ==============================================================================
print("### Step 11: Correlation Scenarios & Final Capital ###\n")
print("Applying user-specified logic: Square first, then apply scenario adjustments.\n")

# 1. Determine Scenario Curvature Correlations
curvature_rho_high = curvature_rho_medium * 1.25
curvature_rho_low = max(2 * curvature_rho_medium - 1, 0.75 * curvature_rho_medium)

scenarios = {
    'Low': curvature_rho_low,
    'Medium': curvature_rho_medium,
    'High': curvature_rho_high
}

scenario_results_list = []

# 2. Re-run all calculations for each scenario
for name, rho in scenarios.items():

    # Recalculate Bucket Capitals (K_b) and Sums (S_b)
    bucket_results_scen = {}
    for bucket_id, group in net_positions_df.groupby(level='Bucket'):
        _, _, k_final, scenario = calculate_bucket_capital(group, rho)
        bucket_df = net_positions_df.loc[bucket_id]
        s_b = bucket_df['CVR+'].sum() if scenario == 'Upward' else bucket_df['CVR-'].sum()
        bucket_results_scen[bucket_id] = {'K_b': k_final, 'S_b': s_b}

    # Recalculate Final RCCR
    sum_sq_kb_scen = sum(res['K_b']**2 for res in bucket_results_scen.values())
    # Cross-bucket correlation term remains 0 for all scenarios in this risk class
    rccr_final = np.sqrt(max(0, sum_sq_kb_scen))

    scenario_results_list.append({
        'Scenario': name,
        'Intra-Bucket Corr': rho,
        'Total Capital (RCCR)': rccr_final
    })

scenario_df = pd.DataFrame(scenario_results_list).set_index('Scenario')
scenario_df['Intra-Bucket Corr'] = scenario_df['Intra-Bucket Corr'].map('{:.2%}'.format)

display_df(scenario_df, "Scenario Capital Results")

# 3. Determine Final Capital Charge
final_capital_charge = scenario_df['Total Capital (RCCR)'].max()
winning_scenario = scenario_df['Total Capital (RCCR)'].idxmax()

print("### Final CSR Curvature Capital Requirement ###\n")
print(f"The final capital charge is the maximum of the three scenarios, which is from the '{winning_scenario}' scenario.\n")
print(f"Final Capital Charge = {final_capital_charge:,.2f}")

### Steps 3 & 4: Establish Gross and Net Positions ###

Each tranche is a unique risk factor, so no netting is required.

--- Final Net Positions ---
                       CVR+  CVR-
Bucket Risk Factor               
3      RMBS Tranche A   750  -600
       RMBS Tranche B  -400   550
       RMBS Tranche C   250  -150
14     ABS Tranche 1   -800   950
       ABS Tranche 2    300  -200
       ABS Tranche 3   -100   450


### Steps 5 & 6: Determine Base Curvature Correlations (Medium Scenario) ###

Curvature correlations are the square of the corresponding delta correlations (Article 325ay(5)).

Base Intra-Bucket Delta Correlation (rho_delta): 40.00%
Base Intra-Bucket Curvature Correlation (rho_curvature): 0.1600 or 16.00%

Base Cross-Bucket Delta Correlation (gamma_delta): 0.00%
Base Cross-Bucket Curvature Correlation (gamma_curvature): 0.0000 or 0.00%


### Steps 7 & 8: Bucket-Level Capital Calculation (Medium Scenario) ###

Calculating for Bucket 3...
  K_b+ = 746.32
  K_b- = 412.92
 